In [1]:
import pandas as pd
import numpy as np
import logging
from config import TABLE_LIST
import pymysql
from datetime import datetime
from dotenv import load_dotenv
import os 


import requests
import pyarrow.parquet as pq
import pyarrow as pa
from sqlalchemy import create_engine, text
from sqlalchemy.pool import QueuePool

In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

### At first we use the PyMysql engine to extract data and load the data directly to file

First I configure mysql / mariadb to accept connections from any IP (could be dangerous) in /etc/mysq/mariadb.conf.d/50-server.cnf 


bind-address = 0.0.0.0

Then I create a read-only user and grant SELECT privileges only to that user for the database whose data is to be extracted:

```CREATE USER 'etl_user'@'%' IDENTIFIED BY 'mypass980898'; FLUSH PRIVILEGES;```

```GRANT SELECT ON database_name.* TO 'etl_user'@'%';```

In [3]:
load_dotenv(".env")

host = os.getenv("HOST")
port = os.getenv("PORT")
user = os.getenv("DB_USER")
db = os.getenv("DB_NAME")
password = os.getenv("PASSWORD")
api_key = os.getenv("API_KEY")
api_secret = os.getenv("API_SECRET")


In [4]:
def test_get_customer():
    BASE_URL = "https://tst.neviraminerals.com/"

    headers = {
        "Authorization": f"token {api_key}:{api_secret}",
        "Content": "application/json"
    }

    params = {
        "page": 1,
        "page_length": 40
    }

    URL = f"{BASE_URL}api/method/neviraflow.api.get_customer_list"
    response = requests.get(URL,params = params, headers=headers)
    if response.status_code == 200:
        print(response)
    
        data = response.json()
        return data
    else:
        print("Failed to connect to URL")

In [5]:
res = test_get_customer()

<Response [200]>


In [6]:
list_of_dicts = res["message"]["data"]
df_res = pd.DataFrame(list_of_dicts)

In [7]:
df_res_ = pd.json_normalize(list_of_dicts)

In [8]:
def get_one_dict(list_dict: list):
    keys = list_of_dicts[0].keys()
    out_dict = {key:[] for key in keys}
    for dict_ in list_of_dicts:
        for key, value in dict_.items():
            out_dict[key].append(value)
    return out_dict

In [9]:
logging.info("Connecting to remote database ....")
connection_config_remote = {
    "host": host,
    "user": user,
    "database":db,
    "port": int(port),
    "password":password
}

## pymysql does not privide connection pooling so we have to open the connection and close it
## every time we need to make a database connection
with pymysql.connect(**connection_config_remote) as connection:
    query = """ SELECT 
                    name, 
                    customer, 
                    customer_name, 
                    posting_date, 
                    due_date, 
                    base_grand_total FROM `tabSales Invoice` WHERE docstatus = 1 
                    AND YEAR(posting_date) = 2026 """
    logging.info("Extracting data ....")
    df_remote = pd.read_sql(query, connection)
    today_date = datetime.strftime(datetime.today(), "%Y-%m-%d %HH-%MM-%SS")
    logging.info("Loading data to file storage")
    df_remote.to_parquet(f"data/sales_invoice_{today_date}")

2026-09-11 15:54:22,792 - INFO - Connecting to remote database ....
2026-09-11 15:54:24,008 - INFO - Extracting data ....
/tmp/ipykernel_64524/3248204510.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_remote = pd.read_sql(query, connection)
2026-09-11 15:54:24,797 - INFO - Loading data to file storage


In [10]:
df_remote.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2112 entries, 0 to 2111
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              2112 non-null   object 
 1   customer          2112 non-null   object 
 2   customer_name     2112 non-null   object 
 3   posting_date      2112 non-null   object 
 4   due_date          2112 non-null   object 
 5   base_grand_total  2112 non-null   float64
dtypes: float64(1), object(5)
memory usage: 99.1+ KB


### Now we move towards extracting the data using SQLAlchemy

SQLAlchemy helps us manage database connections, create an abstraction layer, handles lazy loading and relationship management.

In [11]:
## database connection manager / factory to handle connection to a database i.e mysql, oracle, postgresql

## create engine with connection pooling
mysql_engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{db}",
                            poolclass = QueuePool,
                            pool_size=10, 
                            pool_pre_ping=True, 
                            echo=False
                        )

query = text("""SELECT 
                    name, 
                    supplier, 
                    supplier_name, 
                    posting_date, 
                    due_date, 
                    base_grand_total FROM `tabPurchase Invoice` WHERE docstatus = 1 
                    AND YEAR(posting_date) = 2026 """)

with mysql_engine.connect() as connection:
    df_a = pd.read_sql(query, connection)
    extraction_date = datetime.strftime(datetime.today(), "%Y-%m-%d")
    df_a.to_parquet(f"data/purchase_invoice_{extraction_date}")

In [12]:
datetime.strftime(datetime.now(),"%Y_%m_%d_%H_%M_%S")

'2026_09_11_15_54_47'

Chunking reduces the amount of data held in memory at one given time,
but it does not reduce the total amount of data extracted.

Improvements will need to made for the following scenarios:

- When we have a new table and it has never been extracted yet
- Doing incremental loading while making sure that the new tables are not affected by the date watermark

In [14]:
erp_table_names = TABLE_LIST
with mysql_engine.connect() as connection:
    for erp_table in erp_table_names:
        select_query = text(f""" SELECT * FROM `tab{erp_table}` """)

        table_name_clean = erp_table.replace(" ","")

        table_folder = f"data/{table_name_clean}"

        ## we want to have a sub-directory for every table
        os.makedirs(table_folder, exist_ok=True)
        
        logging.info(f" Extracting data from {erp_table} ..... ")
        extraction_date_time = datetime.strftime(datetime.now(),"%Y_%m_%d_%H_%M_%S")

        parquet_path = f"{table_folder}/{extraction_date_time}.parquet"

        ## read the data in tables in batches instead of everything at once
        chunks = pd.read_sql(select_query,
                            connection, 
                            chunksize=10000 ## meaning we will extract only 10,000 rows at a time
                        )

        parquet_writer = None

        n_rows = 0

        try:
            for chunk in chunks:
                ## Use the schema of the first chunk to create the parquet writer

                if parquet_writer is None:
                    parquet_schema = pa.Table.from_pandas(chunk).schema
                    parquet_writer = pq.ParquetWriter(parquet_path, parquet_schema)

                parquet_table = pa.Table.from_pandas(chunk)
                parquet_writer.write(parquet_table)
                n_rows += len(chunk)

                logger.info(f"Written a total of {n_rows} rows for table {table_name_clean}")

                if n_rows == 0:
                    logger.warning("No rows returned by the query!")
        except Exception as e:
            logger.error(f"An error occured during data extraction ....{e}")

        ## Close the parquet writer
        finally:
            if parquet_writer is not None:
                parquet_writer.close()

    execution_end_time = datetime.now()

    ## Save the execution end time to a file, we will use it for incremental loading the next run
    with open("execution_reads.txt","a") as execution_file:
        execution_file.write(f"{execution_end_time} \n")

2026-09-11 15:55:11,725 - INFO -  Extracting data from Customer ..... 
2026-09-11 15:55:12,412 - INFO - Written a total of 339 rows for table Customer
2026-09-11 15:55:12,415 - INFO -  Extracting data from Sales Team ..... 
2026-09-11 15:55:13,266 - INFO - Written a total of 5081 rows for table SalesTeam
2026-09-11 15:55:13,268 - INFO -  Extracting data from Supplier ..... 
2026-09-11 15:55:13,613 - INFO - Written a total of 124 rows for table Supplier
2026-09-11 15:55:13,617 - INFO -  Extracting data from Contact ..... 
2026-09-11 15:55:13,922 - INFO - Written a total of 216 rows for table Contact
2026-09-11 15:55:13,926 - INFO -  Extracting data from Address ..... 
2026-09-11 15:55:14,192 - INFO - Written a total of 254 rows for table Address
2026-09-11 15:55:14,195 - INFO -  Extracting data from Employee ..... 
2026-09-11 15:55:14,441 - INFO - Written a total of 0 rows for table Employee
2026-09-11 15:55:14,443 - WARNING - No rows returned by the query!
2026-09-11 15:55:14,446 - INF

ProgrammingError: (pymysql.err.ProgrammingError) (1146, "Table '_8a723ea3d5c7c0d5.tabJournal Entry Reference' doesn't exist")
[SQL:  SELECT * FROM `tabJournal Entry Reference` ]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [43]:
execution_end_time = datetime.now()
with open("execution_reads.txt","a") as execution_file:
    execution_file.write(f"{execution_end_time}\n")

In [20]:
df_a.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308 entries, 0 to 307
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              308 non-null    object 
 1   supplier          308 non-null    object 
 2   supplier_name     308 non-null    object 
 3   posting_date      308 non-null    object 
 4   due_date          308 non-null    object 
 5   base_grand_total  308 non-null    float64
dtypes: float64(1), object(5)
memory usage: 14.6+ KB


In [ ]:
### Parametized query to get data using parameters
def get_invoice_by_customer(customer_id):
    query = text("""
                    SELECT si.name,
                           si.customer_name,
                           si.posting_date, 
                           si.due_date,
                           si.payment_terms_template,
                           si.base_grand_total,
                           si.outstanding_amount,
                           si.status
                    FROM `tabSales Invoice` AS si 
                    WHERE si.customer =:customer_id
                    AND si.docstatus = 1
                    ORDER BY si.posting_date DESC
                """)


    ## Context manager 
    with mysql_engine.connect() as connection:
        
        result = connection.execute(query, {"customer_id":customer_id}) ## Applying a filter to the query
        df = pd.DataFrame(result.fetchall(), columns = result.keys())
    return df
        
    

In [24]:
res = get_invoice_by_customer("ASL PACKAGING LTD")
res

,name,customer_name,posting_date,due_date,payment_terms_template,base_grand_total,outstanding_amount,status
0,ACC-SINV-2026-02107,ASL PACKAGING LTD,2026-08-15,2026-11-13,90 Days,140360.000000000,140360.000000000,Unpaid
1,ACC-SINV-2026-02090,ASL PACKAGING LTD,2026-08-13,2026-11-11,90 Days,95120.000000000,95120.000000000,Unpaid
2,ACC-SINV-2026-02070,ASL PACKAGING LTD,2026-08-12,2026-11-10,90 Days,20416.000000000,20416.000000000,Unpaid
3,ACC-SINV-2026-02057,ASL PACKAGING LTD,2026-08-11,2026-11-09,90 Days,60784.000000000,60784.000000000,Unpaid
4,ACC-SINV-2026-02058,ASL PACKAGING LTD,2026-08-11,2026-11-09,90 Days,82592.000000000,82592.000000000,Unpaid
...,...,...,...,...,...,...,...,...
247,ACC-SINV-2024-00360,ASL PACKAGING LTD,2024-06-22,2024-09-20,90 Days,74704.000000000,0E-9,Paid
248,ACC-SINV-2024-00358-1,ASL PACKAGING LTD,2024-06-22,2024-09-20,90 Days,65888.000000000,0E-9,Paid
249,ACC-SINV-2024-00158,ASL PACKAGING LTD,2024-05-22,2024-08-20,90 Days,28536.000000000,0E-9,Paid
250,ACC-SINV-2024-00097,ASL PACKAGING LTD,2024-05-14,2024-08-12,90 Days,92800.000000000,0E-9,Paid


In [25]:
def individual_sales_report():
    query = text(""" 
                    SELECT 
                        si.name AS invoice_number,
                        si.posting_date,
                        si.customer,
                        si.customer_name,
                        st.sales_person,
                        si.due_date,
                        si.currency,
                        si.payment_terms_template,
                        sii.item_code,
                        sii.item_name, 
                        sii.rate,
                        sii.qty,
                        ROUND(si.base_grand_total,2) AS base_grand_total,
                        ROUND(si.outstanding_amount,2) AS outstanding_amount,
                        si.status
                    FROM `tabSales Invoice Item` AS sii 
                        INNER JOIN `tabSales Invoice` AS si ON sii.parent = si.name
                        LEFT JOIN `tabSales Team` AS st ON si.customer = st.parent
                    WHERE si.docstatus = 1 
                        AND si.is_opening = 0
                        AND si.is_return = 0
                        AND si.status NOT IN ('Credit Note Issued')
                    ORDER BY si.posting_date DESC
                    """)
    with engine.connect() as connection:
        df = pd.read_sql(query, connection)
        
    if not df.empty:
        ## transformation step
        df['amount_paid'] = df['base_grand_total'] -  df['outstanding_amount']

    return df

In [26]:
item_wise_sales = individual_sales_report()

In [27]:
item_wise_sales.sort_values(by='amount_paid',ascending=False).query("status == 'Paid' ").head(5)

,invoice_number,posting_date,customer,customer_name,sales_person,due_date,currency,payment_terms_template,item_code,item_name,rate,qty,base_grand_total,outstanding_amount,status,amount_paid
3570,ACC-SINV-2026-00302,2026-02-12,COLOURFLEX INKS &COATINGS LTD,COLOURFLEX INKS &COATINGS LTD,David,2026-02-13,KES,1 Day,NC3022,Nc Yellow H/P Concentrate-180kgs,680.0,360.0,4758000.0,0.0,Paid,4758000.0
3569,ACC-SINV-2026-00302,2026-02-12,COLOURFLEX INKS &COATINGS LTD,COLOURFLEX INKS &COATINGS LTD,David,2026-02-13,KES,1 Day,NCB2014,Nc Black H/P Concentrate-180kgs,650.0,180.0,4758000.0,0.0,Paid,4758000.0
3566,ACC-SINV-2026-00302,2026-02-12,COLOURFLEX INKS &COATINGS LTD,COLOURFLEX INKS &COATINGS LTD,David,2026-02-13,KES,1 Day,NC3021,Nc Yellow H/P Concentrate-182kgs,680.0,3640.0,4758000.0,0.0,Paid,4758000.0
3567,ACC-SINV-2026-00302,2026-02-12,COLOURFLEX INKS &COATINGS LTD,COLOURFLEX INKS &COATINGS LTD,David,2026-02-13,KES,1 Day,RN320,Polyamide Resin-211,615.0,1200.0,4758000.0,0.0,Paid,4758000.0
3568,ACC-SINV-2026-00302,2026-02-12,COLOURFLEX INKS &COATINGS LTD,COLOURFLEX INKS &COATINGS LTD,David,2026-02-13,KES,1 Day,NCB2013,Nc Black H/B Concentrate-182kgs,650.0,1820.0,4758000.0,0.0,Paid,4758000.0


### Extracting large databases / tables with batch processing and exception handling

In [28]:
import logging
from sqlalchemy.exc import SQLAlchemyError, OperationalError

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def fetch_stock_ledger_entry():
    query = text(""" SELECT * FROM `tabStock Ledger Entry` WHERE is_cancelled = 0 ORDER BY creation DESC""")

    try:
        with engine.connect() as connection:
            results = connection.execution_options(stream_results=True).execute(query)
    
            ## Process the data in chunks
            chunk_size = 10000
    
            chunks = []
    
            while True:
                chunk = results.fetchmany(chunk_size)
    
                if not chunk:
                    break
                chunks.append(pd.DataFrame(chunk, columns = results.keys()))
            
            sle_df =  pd.concat(chunks, ignore_index=True)

            ## get the query execution time
            run_time = datetime.strftime(datetime.today(), "%Y-%m-%d")
            
            ## Save dataframe to parquet
            sle_df.to_parquet(f"data/stock_ledger_entry_{run_time}.parquet")

            return sle_df
            
            
    except OperationalError as e:
        logger.error(f"Database operation error: {e}")
        return None
        
    except SQLAlchemyError as e:
        logger.error(f"SQLAlchemy error: {e}")
        return None

    except Exception as e:
        logger.error(f"Unexected error occured: {e}")
        return None
        

In [29]:
sle_df = fetch_stock_ledger_entry()

In [30]:
sle_df.shape

(174717, 45)

In [31]:
query = text(""" SELECT table_name FROM information_schema.tables WHERE table_schema = 'erpnext_db' AND table_name NOT IN  ('__UserSettings','__global_search','__Auth') ORDER BY table_name ASC; """)